In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lhungen@gmail.com/FMCG_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
print(catalog, data_source)

fmcg customers


In [0]:
base_path = f's3://fmcg-databricks-bronze/{data_source}/*.csv'
print(base_path)

s3://fmcg-databricks-bronze/customers/*.csv


## Bronze

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    # Add read_timestamp and file_name to record when the data was ingested and from where
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-03-19T17:48:43.379Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-03-19T17:48:43.379Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-03-19T17:48:43.379Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-03-19T17:48:43.379Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-03-19T17:48:43.379Z,customers.csv,1404


In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-03-19 17:49:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-03-19 17:49:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-03-19 17:49:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-03-19 17:49:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-03-19 17:49:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-03-19 17:49:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-03-19 17:49:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-03-19 17:49:...|customers.csv|     1404|
|     7891

In [0]:
df_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



Transformation

- 1: Drop Duplicates

In [0]:
## remove duplicate
print("Number of rows before dropping duplicate", df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print("Number of rows after dropping duplicate", df_silver.count())

Number of rows before dropping duplicate 39
Number of rows after dropping duplicate 35


- 2: Trim spaces in customer name

In [0]:
#remove space from customer name

## to show the columns with space issues first
print("Before trim")
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

## remove the space
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

## check if everything is cleaned
print("After trim")
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

Before trim


customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-03-19T17:49:01.646Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-03-19T17:49:01.646Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-03-19T17:49:01.646Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-03-19T17:49:01.646Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-03-19T17:49:01.646Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-03-19T17:49:01.646Z,customers.csv,1404


After trim


customer_id,customer_name,city,read_timestamp,file_name,file_size


- 3: Data Quality Fix: Correcting City Typos

In [0]:
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
df_silver.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bengaluru|
|Hyderabad|
|New Delhi|
|     NULL|
+---------+



- 4: Fix Title-Casing Issue

In [0]:
df_silver.select("customer_name").distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      FitFuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|HydroBoost Nutrition|
|MacroBite Superfoods|
|MacroBite superfoods|
|      PowerSnack Hub|
|      PowerSnack hub|
|   SprintX nutrition|
|   SprintX Nutrition|
|    ZenAthlete foods|
|    ZenAthlete Foods|
|Peak performance ...|
|Peak Performance ...|
| PrimeFuel Nutrition|
|       Recovery Lane|
|      StaminaX Store|
|EliteAthlete Nutr...|
|      GamePlan Foods|
|   Champion's choice|
+--------------------+
only showing top 20 rows


In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

In [0]:
df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



- 5: Handling missing cities

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

+-----------+-------------------+----+--------------------------+-------------+---------+
|customer_id|customer_name      |city|read_timestamp            |file_name    |file_size|
+-----------+-------------------+----+--------------------------+-------------+---------+
|789403     |Sprintx Nutrition  |NULL|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789603     |Recovery Lane      |NULL|2026-03-19 17:49:01.646904|customers.csv|1404     |
+-----------+-------------------+----+--------------------------+-------------+---------+



In [0]:
null_customer_names = ["Sprintx Nutrition", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timestamp            |file_name    |file_size|
+-----------+-------------------+---------+--------------------------+-------------+---------+
|789401     |Sprintx Nutrition  |Bengaluru|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789402     |Sprintx Nutrition  |Hyderabad|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789403     |Sprintx Nutrition  |NULL     |2026-03-19 17:49:01.646904|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL     |2026-03-19 17:49:01.646904|customers.csv|1404     |
|789421     |Zenathlete Foods   |Hyderabad|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789422     |Zenathlete Foods   |New Delhi|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789520     |Primefuel Nutrition|Bengaluru|2026-03-19 17:49:01.646904|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL     |2026-03

In [0]:
# the city for the same customer name is different, not sure which location those null data is. After confirming with business owner:
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
#merged the fixed city into the main dataset
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
df_silver.filter(F.col("customer_name").isNull()).show()

+-----------+-------------+----+--------------+---------+---------+
|customer_id|customer_name|city|read_timestamp|file_name|file_size|
+-----------+-------------+----+--------------+---------+---------+
+-----------+-------------+----+--------------+---------+---------+



All null data flixed

- 6: Convert customer_id to string

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-03-19T17:49:01.646Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-03-19T17:49:01.646Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-03-19T17:49:01.646Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-03-19T17:49:01.646Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-03-19T17:49:01.646Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold Layer

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

In [0]:
# We don't need all columns for golden layer, so we just select the ones we need
# In the parent model, the model is customer_code(customer_id), customer, market, platform , channel
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

## Merging Data source with parent

In [0]:
parent_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
parent_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]